# Phishing Email Detection with Machine Learning

A cybersecurity machine learning workflow for detecting phishing emails using TF-IDF features, Random Forest, Support Vector Machine, and Logistic Regression.

## What this notebook demonstrates

- Load and clean a labeled phishing email dataset.
- Vectorize email text with TF-IDF.
- Train and compare Random Forest, SVM, and Logistic Regression classifiers.
- Evaluate models with accuracy, precision, recall, F1-score, and confusion matrices.
- Analyze important terms that contribute to phishing detection.



# Enhancing Phishing Email Detection Using Real Datasets

### Introduction
This project aims to build and train several machine learning models to distinguish between secure emails and phishing messages. We will perform all the basic steps, from processing text data to evaluating and selecting the best-performing model.

### Loading Libraries and Data
In the first step, we will import all the libraries we will need for the project. We will then load the [dataset](https://www.kaggle.com/datasets/subhajournal/phishingemails) from a CSV file and preview it to understand its structure and size.


In [ ]:
# Connect Google Drive
# Optional for Google Colab users:
# from google.colab import drive
# drive.mount('/content/drive')


# Import Necessary Libraries


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.pyplot as plt


# Load the Dataset


In [ ]:
# Define the file path for the dataset
file_path = 'data/Phishing_Email.csv'

try:
    # Read the CSV file
    df = pd.read_csv(file_path)
except FileNotFoundError:
    print(f"Error: File not found at path '{file_path}'.")


# Data Exploration


In [ ]:
# Display the first 5 rows and concise summary of the DataFrame
print(df.info())
df.head()


After loading the data, it's important to understand its nature and distribution. At this stage, we'll analyze the number of emails of each type (safe and fraudulent) to see if the dataset is balanced. This will provide us with a basic overview of the data we'll be working with.


In [ ]:
# # Check the distribtion of emails types
df['Email Type'].value_counts()


In [ ]:
# # Visualize the distribution of email types
plt.figure(figsize=(8, 5))
sns.countplot(x='Email Type', data=df, palette='viridis', hue='Email Type', legend=False)
plt.title('Email Distribution (Fraudulent vs. Legitimate)')
plt.xlabel('Email Type')
plt.ylabel('Number')
plt.show()


# Data Cleaning


Basic steps to clean and clean data:
* **Deal with missing values:** Delete any rows that do not contain email text.
* **Remove junk data:** Eliminate unnecessary columns that do not contribute to the analysis.
* **Delete duplicates:** Remove any duplicate emails entirely to prevent bias in the model.


In [ ]:
# Check for missing values before cleaning
print(f"Number of missing values ​​in 'Email Text' before cleanup: {df['Email Text'].isnull().sum()}")

# Handle Missing Values
df.dropna(subset=['Email Text'], inplace=True)

# Remove Redundant Columns
if 'Unnamed: 0' in df.columns:
    df.drop('Unnamed: 0', axis=1, inplace=True)

# Handle Duplicate Rows

# Check for and count duplicate rows before removal
num_duplicates = df.duplicated().sum()
print(f"Number of duplicate rows before cleaning: {num_duplicates}")

# Drop duplicate rows from the DataFrame
df.drop_duplicates(inplace=True)


In [ ]:
# Verification After Cleaning
print(f"Number of rows remaining: {len(df)}")
print(df.info())
df.head()


# Text Preprocessing and Data Splitting


Text to Vectorization: Models cannot directly understand text, so we use TF-IDF to convert each email message into a numerical vector representing the importance of the words it contains.

Data Splitting: We split the data into two parts: a training set (80%) to teach the model, and a test set (20%) to evaluate its performance on new data it has never seen before.


In [ ]:
# Define Features (X) and Target (y)
X = df['Email Text']
y = df['Email Type']

# Encode the Target Variable
# Convert Safe Email, Phishing Email into (0, 1)
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)

# Vectorize the Text Data (X) using TF-IDF
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
X_vectorized = vectorizer.fit_transform(X)

# Split Data into Training and Testing Sets
# 80% for training and 20% for testing
X_train, X_test, y_train, y_test = train_test_split(X_vectorized,y_encoded,test_size=0.2,random_state=42,
    stratify=y_encoded
)

# Verify the shape of the split data
print("Data sizes after splitting")
print(f" Training data size (X_train): {X_train.shape}")
print(f" Test data size (X_test): {X_test.shape}")
print(f" Training data size (y_train): {y_train.shape}")
print(f" Test data size (y_test): {y_test.shape}")


# Training models and evaluating their performance


We will train three different machine learning models and compare their performance. The models are:

1. **Random Forest:** A robust model based on multiple decision trees.
2. **Support Vector Machine (SVM):** A model that is effective in high-dimensional classification tasks.
3. **Logistic Regression:** A classic and reliable model used as a baseline for comparison.

We will evaluate each model using accuracy, precision, recall, and F1-score metrics, as well as a confusion matrix to visualize its performance.


## Random Forest Classifier


In [ ]:
# Initialize the model
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

# Train the model
rf_model.fit(X_train, y_train)

# Make predictions
y_pred_rf = rf_model.predict(X_test)

# Model Evaluation
print(f"Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf, target_names=['Safe Email', 'Phishing Email']))

# display the confusion matrix
cm_rf = confusion_matrix(y_test, y_pred_rf)
disp_rf = ConfusionMatrixDisplay(confusion_matrix=cm_rf, display_labels=['Safe Email', 'Phishing Email'])
disp_rf.plot(cmap=plt.cm.Blues)
plt.title("Confusion Matrix - Random Forest")
plt.show()


## Support Vector Machine - SVM


In [ ]:
# Initialize the model
svm_model = SVC(kernel='linear', random_state=42) # 'linear' kernel is a good start

# Train the model
svm_model.fit(X_train, y_train)

# Make predictions
y_pred_svm = svm_model.predict(X_test)

# Model Evaluation
print(f"Accuracy: {accuracy_score(y_test, y_pred_svm):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_svm, target_names=['Safe Email', 'Phishing Email']))

# display the confusion matrix
cm_svm = confusion_matrix(y_test, y_pred_svm)
disp_svm = ConfusionMatrixDisplay(confusion_matrix=cm_svm, display_labels=['Safe Email', 'Phishing Email'])
disp_svm.plot(cmap=plt.cm.Greens)
plt.title("Confusion Matrix - SVM")
plt.show()


## Logistic Regression


In [ ]:
# Initialize the model
lr_model = LogisticRegression(max_iter=1000, random_state=42)

# Train the model
lr_model.fit(X_train, y_train)

# Make predictions
y_pred_lr = lr_model.predict(X_test)

# Model Evaluation
print(f"Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr, target_names=['Safe Email', 'Phishing Email']))

# display the confusion matrix
cm_lr = confusion_matrix(y_test, y_pred_lr)
disp_lr = ConfusionMatrixDisplay(confusion_matrix=cm_lr, display_labels=['Safe Email', 'Phishing Email'])
disp_lr.plot(cmap=plt.cm.Oranges)
plt.title("Confusion Matrix - Logistic Regression")
plt.show()


### Model Performance Analysis
All models achieved very strong performance, but the SVM model is considered the best, due to its highest overall accuracy while maintaining excellent phishing detection metrics (98% recall). This means it best balances threat detection with reducing false positives.


# Optimization of the Best Model (SVM)


Optimization: After selecting SVM as the best model, we tested different kernels (linear vs. rbf).

Both achieved excellent results (97.6% vs. 97.7% accuracy).

The linear kernel was preferred for its simplicity and speed, confirming the robustness of the chosen model.


In [ ]:
# Improvement on SVM - Comparing kernel linear and rbf
for kernel in ['linear', 'rbf']:
    svm_model = SVC(kernel=kernel, C=1, random_state=42)
    svm_model.fit(X_train, y_train)
    y_pred = svm_model.predict(X_test)
    print(f"Kernel: {kernel}")
    print("Accuracy:", accuracy_score(y_test, y_pred))


---


# Feature Importance Analysis


Here, we will analyze the features (words) that the Random Forest model found most important in distinguishing between phishing and secure messages.

This analysis helps us better understand the nature of phishing messages.


In [ ]:
# Extract feature importances from the trained Random Forest model
feature_importances = rf_model.feature_importances_

# names of the features from the TF-IDF vectorizer
feature_names = vectorizer.get_feature_names_out()

# DataFrame to view features and their importance scores
features_df = pd.DataFrame({'feature': feature_names, 'importance': feature_importances})

# Sort the features by importance in descending order
features_df = features_df.sort_values(by='importance', ascending=False)

# most important features
top_features = features_df.head(15)
print("Top 15 Phishing Words")
print(top_features)


# Visualize the Top Features
plt.figure(figsize=(10, 8))
plt.barh(top_features['feature'], top_features['importance'], color='skyblue')
plt.gca().invert_yaxis()
plt.title('Top 15 Features (Words) for Detecting Phishing Messages')
plt.xlabel('Importance')
plt.show()


# Model performance comparison


In [ ]:
# Calculate accuracy scores
acc_svm = accuracy_score(y_test, y_pred_svm)
acc_lr = accuracy_score(y_test, y_pred_lr)
acc_rf = accuracy_score(y_test, y_pred_rf)

# dictionary with model names and their calculated accuracy scores
model_performance = {
    'SVM': acc_svm,
    'Logistic Regression': acc_lr,
    'Random Forest': acc_rf
}

# Convert the dictionary to a DataFrame for plotting
perf_df = pd.DataFrame(list(model_performance.items()), columns=['Model', 'Accuracy'])
perf_df = perf_df.sort_values(by='Accuracy', ascending=False)

# Bar Plot
plt.figure(figsize=(8, 6))
ax = sns.barplot(data=perf_df, x='Model', y='Accuracy', hue='Model', palette='mako', legend=False)

# accuracy value on top of each bar
for p in ax.patches:
    ax.annotate(f'{p.get_height():.4f}',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center',
                xytext=(0, 9),
                textcoords='offset points')

plt.title('Model Performance Comparison')
plt.ylabel('Accuracy')
plt.xlabel('Model')
plt.ylim(min(model_performance.values()) - 0.005, max(model_performance.values()) + 0.005)
plt.show()


As the chart above shows, the SVM model performed the best, with an accuracy of 97.63%. Based on these comprehensive results, the SVM model was selected as the final recommended model for application in an anti-phishing system.
